# 22 - Original Language Classification

Train and compare multiclass Logistic Regression models to predict a movie's original language. The dataset is limited to its three most frequent language classes before a stratified 70/15/15 train, validation, and test split. Validation macro F1 selects the configuration; the test set is reserved for one final evaluation.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_PATH = PROJECT_ROOT / "data" / "features" / "11_language_features.csv"
TARGET = "target_original_language"
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH, low_memory=False)
language_counts = df[TARGET].value_counts()
top_languages = language_counts.head(3).index.tolist()
model_df = df.loc[df[TARGET].isin(top_languages)].copy()

print(f"Source shape: {df.shape}")
print(f"Top 3 languages: {top_languages}")
display(model_df[TARGET].value_counts().rename_axis("language").to_frame("movies"))

Source shape: (45422, 33)
Top 3 languages: ['en', 'fr', 'it']


,movies
language,
en,32251
fr,2436
it,1529


## Features and Splits

`id` and the target are excluded. Numeric production and text-length features are scaled; `status` is one-hot encoded; original-title character n-grams, genre terms, and production-country terms are TF-IDF encoded. Raw `original_language` is not present in this dataset, and high-cardinality fields such as title alternatives, cast, director, companies, and overview are excluded.

In [3]:
numeric_features = [
    "release_year", "release_month", "release_decade",
    "log_budget", "log_revenue", "runtime", "log_popularity",
    "genre_count", "has_collection", "company_count",
    "country_count", "cast_size", "crew_size", "has_director",
    "keyword_count", "title_word_count", "overview_word_count",
    "title_differs_from_original", "has_tagline",
    "budget_known", "revenue_known",
]
categorical_features = ["status"]
text_features = ["original_title", "genres", "production_countries"]
feature_columns = numeric_features + categorical_features + text_features

missing_columns = sorted(set(feature_columns + [TARGET]) - set(model_df.columns))
if missing_columns:
    raise KeyError(f"Expected columns are missing from the feature dataset: {missing_columns}")

X = model_df[feature_columns].copy()
for column in text_features:
    X[column] = X[column].fillna("").astype(str)

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(model_df[TARGET])
language_names = target_encoder.classes_

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

split_summary = pd.DataFrame(
    {"rows": [len(X_train), len(X_validation), len(X_test)]},
    index=["Training", "Validation", "Test"],
)
display(split_summary)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
status_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("status", status_pipeline, categorical_features),
        ("original_title", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2, max_features=20000, sublinear_tf=True), "original_title"),
        ("genres", TfidfVectorizer(ngram_range=(1, 2), min_df=2), "genres"),
        ("production_countries", TfidfVectorizer(ngram_range=(1, 2), min_df=2), "production_countries"),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_validation_encoded = preprocessor.transform(X_validation)
X_test_encoded = preprocessor.transform(X_test)
print(f"Encoded feature matrix: {X_train_encoded.shape[1]:,} features")

,rows
Training,25351
Validation,5432
Test,5433


Encoded feature matrix: 20,910 features


## Validation Model Comparison

Macro-averaged precision, recall, and F1 give each language equal weight despite the class imbalance. Select the model by validation macro F1; use test data only after configuration selection.

In [ ]:
configurations = [
    {"C": c, "solver": "lbfgs", "class_weight": weight}
    for weight in [None, "balanced"]
    for c in [0.1, 1.0, 10.0]
] + [
    {"C": 1.0, "solver": "saga", "class_weight": weight}
    for weight in [None, "balanced"]
]

results = []
trained_models = {}
for model_id, config in enumerate(configurations, start=1):
    model = LogisticRegression(
        C=config["C"],
        solver=config["solver"],
        class_weight=config["class_weight"],
        max_iter=1000,
        tol=1e-3,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_encoded, y_train)
    validation_predictions = model.predict(X_validation_encoded)
    weight_label = config["class_weight"] or "None"
    results.append(
        {
            "model_id": model_id,
            "configuration": f"C={config['C']:g}, {config['solver']}, weight={weight_label}",
            **config,
            "accuracy": accuracy_score(y_validation, validation_predictions),
            "precision_macro": precision_score(y_validation, validation_predictions, average="macro", zero_division=0),
            "recall_macro": recall_score(y_validation, validation_predictions, average="macro", zero_division=0),
            "f1_macro": f1_score(y_validation, validation_predictions, average="macro", zero_division=0),
        }
    )
    trained_models[model_id] = model
    print(f"Finished {model_id}/{len(configurations)}: {results[-1]['configuration']}")

validation_results = pd.DataFrame(results)
display(validation_results.sort_values("f1_macro", ascending=False).round(4))

IndentationError: unexpected indent (304367840.py, line 24)

In [ ]:
plot_results = validation_results.sort_values("f1_macro", ascending=False).reset_index(drop=True)
metric_columns = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
metric_labels = ["Accuracy", "Macro precision", "Macro recall", "Macro F1"]

fig, ax = plt.subplots(figsize=(13, 6))
for column, label in zip(metric_columns, metric_labels):
    ax.plot(plot_results.index, plot_results[column], marker="o", linewidth=1.8, label=label)
ax.set_xticks(plot_results.index, plot_results["configuration"], rotation=40, ha="right")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Logistic Regression configuration")
ax.set_ylabel("Validation score")
ax.set_title("Validation Performance by Hyperparameter Configuration")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
plt.show()

## Select and Evaluate Once on the Test Set

The configuration is chosen using validation macro F1 (accuracy breaks ties). The selected model is now evaluated on the held-out test set; its predictions are reused for the confusion matrix and class-wise F1 chart.

In [ ]:
best_row = validation_results.sort_values(
    ["f1_macro", "accuracy"], ascending=False
).iloc[0]
best_model_id = int(best_row["model_id"])
best_model = trained_models[best_model_id]
best_config = configurations[best_model_id - 1]
print("Selected by validation macro F1:", best_config)

y_test_pred = best_model.predict(X_test_encoded)
test_metrics = {
    "accuracy": accuracy_score(y_test, y_test_pred),
    "precision_macro": precision_score(y_test, y_test_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_test_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_test_pred, average="macro", zero_division=0),
}
display(pd.DataFrame([test_metrics], index=["Test"]).round(4))

test_report = classification_report(
    y_test,
    y_test_pred,
    labels=np.arange(len(language_names)),
    target_names=language_names,
    output_dict=True,
    zero_division=0,
)
per_language_metrics = pd.DataFrame(
    {
        "language": language_names,
        "precision": [test_report[language]["precision"] for language in language_names],
        "recall": [test_report[language]["recall"] for language in language_names],
        "f1": [test_report[language]["f1-score"] for language in language_names],
        "support": [test_report[language]["support"] for language in language_names],
    }
)
display(per_language_metrics.round(4))

In [ ]:
test_confusion = confusion_matrix(y_test, y_test_pred, labels=np.arange(len(language_names)))
display_labels = target_encoder.inverse_transform(np.arange(len(language_names)))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=test_confusion, display_labels=display_labels
).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Test Confusion Matrix: Original Language")
ax.set_xlabel("Predicted language")
ax.set_ylabel("True language")
fig.tight_layout()
plt.show()

In [ ]:
class_f1 = per_language_metrics.sort_values("f1", ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(class_f1["language"], class_f1["f1"], color=["#F58518", "#72B7B2", "#4C78A8"])
ax.bar_label(bars, fmt="%.3f", padding=4)
ax.set_xlim(0, 1.05)
ax.set_xlabel("Test F1-score")
ax.set_ylabel("Original language")
ax.set_title("Test F1-score by Language Class")
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

best_class = per_language_metrics.loc[per_language_metrics["f1"].idxmax()]
hardest_class = per_language_metrics.loc[per_language_metrics["f1"].idxmin()]
print(f"Best predicted class by F1: {best_class['language']} (F1={best_class['f1']:.3f})")
print(f"Most difficult class by F1: {hardest_class['language']} (F1={hardest_class['f1']:.3f})")

## Single-Movie Inference Example

The example below passes one held-out movie's features through the fitted preprocessing and selected classifier, then decodes the predicted language label.

In [ ]:
def predict_original_language(movie_sample):
    transformed_sample = preprocessor.transform(movie_sample[feature_columns])
    encoded_prediction = best_model.predict(transformed_sample)
    return target_encoder.inverse_transform(encoded_prediction)

inference_sample = X_test.iloc[[0]]
inference_prediction = predict_original_language(inference_sample)[0]
true_language = target_encoder.inverse_transform([y_test[0]])[0]
inference_preview = inference_sample[[
    "original_title", "production_countries", "genres", "runtime", "log_popularity"
]].copy()
inference_preview["actual_language"] = true_language
inference_preview["predicted_language"] = inference_prediction
display(inference_preview)